# The objective of this notebook is to test the modularity of the exactBO library

## 0 - Imports

In [1]:
import tamubo.exactbo as ebo
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel

## 1 - ExactBO Loop Class

### 1.1 - Inputs

In [13]:
# Define model
kernel = ConstantKernel(1.0, (1e-3, 1e5)) * RBF(length_scale=0.5, length_scale_bounds=(10, 1e4))
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)

# Define bounds
bounds = [[-20,20],[-20,20]]

# Define precision
precision = 0.05

### 1.2 - Create object

In [14]:
eboloop = ebo.ExactBOLoop(gp, bounds, precision)

### 1.3 - Create true function and set it in object

In [15]:
# Function to minimize
def f(X):
    x, y = X[:,0], X[:,1]

    # Parameters
    alpha = 0.02
    A  = np.array([4.0, 3.0, 2.0])
    B  = np.array([8.0, 5.0, 2.0])    # betas
    C  = np.array([[ 12.0, -3.0],      # centers (x1,y1)
                [-1.0,  15.0],
                [-6.0, -7.0]])
    
    # Compute function value
    val = alpha*(x**2 + y**2)
    for Ai, Bi, (xi, yi) in zip(A, B, C):
        r2 = (x - xi)**2 + (y - yi)**2
        val -= Ai * np.exp(-r2 / Bi)
    return val

# Set in object
eboloop.set_oracle(f)

### 1.4 - Create initial point and evaluate

In [16]:
X0 = np.array([[0,0],[-10,-10],[-10,10],[10,-10],[10,10],[-20,-20],[20,20],[-20,20],[20,-20]])
y0 = f(X0)

### 1.5 - Run optimization (commented to test partition loop)

In [17]:
res = eboloop.run(X0,y0,20)
res

/Users/juanesfco/tamubo/venvs/venvEBO/lib/python3.13/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 17 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
/Users/juanesfco/tamubo/venvs/venvEBO/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/juanesfco/tamubo/venvs/venvEBO/lib/python3.13/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 23 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _c

BOResult(X=array([[  0.   ,   0.   ],
       [-10.   , -10.   ],
       [-10.   ,  10.   ],
       [ 10.   , -10.   ],
       [ 10.   ,  10.   ],
       [-20.   , -20.   ],
       [ 20.   ,  20.   ],
       [-20.   ,  20.   ],
       [ 20.   , -20.   ],
       [  0.625,  -9.375],
       [-19.375,   0.625],
       [  0.625,  -0.625],
       [  0.625,  -0.625],
       [  0.625,  -0.625],
       [  0.625,  -0.625],
       [  0.625,   0.625],
       [ -0.625,  -0.625],
       [  0.625,   0.625],
       [ -0.625,  -0.625],
       [  0.625,   0.625],
       [ -0.625,  -0.625],
       [ -0.625,   0.625],
       [  0.625,  -0.625],
       [  0.625,   0.625],
       [ -0.625,  -0.625],
       [  0.625,   0.625],
       [ -0.625,  -0.625],
       [  0.625,  -0.625],
       [ -0.625,   0.625]]), y=array([-1.97778020e-08,  3.99999255e+00,  4.00000000e+00,  3.99469288e+00,
        4.00000000e+00,  1.60000000e+01,  1.60000000e+01,  1.60000000e+01,
        1.60000000e+01,  1.76562500e+00,  7.51562500

## 2 - Partition Loop Class

### 2.1 - Train model

In [7]:
eboloop.model.fit(X0, y0.ravel())

/Users/juanesfco/tamubo/venvs/venvEBO/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 1.0. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


,kernel,1**2 * RBF(length_scale=0.5)
,alpha,1e-06
,optimizer,'fmin_l_bfgs_b'
,n_restarts_optimizer,0
,normalize_y,True
,copy_X_train,True
,n_targets,None
,random_state,None
,kernel__k1,1**2
,kernel__k2,RBF(length_scale=0.5)
,kernel__k1__constant_value,1.0


### 2.2 - Create partition loop class

In [8]:
ploop = ebo.PartitionMaxEISearch(eboloop.model,eboloop.init_box,eboloop.grid, eboloop.precision)

In [9]:
ploop.boxes, ploop.best_x, ploop.max_ei

([Box(bounds=array([[-20.,  20.],
         [-20.,  20.]]), sampled=True, active=True)],
 None,
 0.0)

### 2.3 - Run a couple iterations of partition loop until it cannot longer partition

In [10]:
ploop.run(1)
len(ploop.boxes), ploop.best_x, ploop.max_ei

(4, array([-20.,  20.]), np.float64(0.01358068304274647))

In [11]:
ploop.run(1)
len(ploop.boxes), ploop.best_x, ploop.max_ei

(16, array([ 5., -5.]), np.float64(0.013580683043505737))

In [12]:
ploop.run(1)
len(ploop.boxes), ploop.best_x, ploop.max_ei

(64, array([-2.5, -2.5]), np.float64(0.013721657794326256))

In [13]:
ploop.run(1)
len(ploop.boxes), ploop.best_x, ploop.max_ei

(256, array([-1.25, -1.25]), np.float64(0.03494242889846291))

In [14]:
ploop.run(1)
len(ploop.boxes), ploop.best_x, ploop.max_ei

(1024, array([-0.625, -0.625]), np.float64(0.12305104912315462))

In [15]:
ploop.run(1)
len(ploop.boxes), ploop.best_x, ploop.max_ei

(1024, array([-0.625, -0.625]), np.float64(0.12305104912315462))